# 01 — Data Ingestion

## Goals
- Load raw M5 CSV files in chunks to avoid RAM overflow
- Convert wide-format sales data to long-format (one row per product per day)
- Join with calendar and price data
- Write final dataset to Parquet on disk for efficient access in downstream notebooks

## Why Parquet
- Columnar format — read only the columns you need without loading the whole file
- Compressed — ~4x smaller than CSV on disk
- Fast — optimized for analytical read patterns
- Industry standard for large analytical datasets

## Data Files
- `sales_train_validation.csv` — 30,490 rows × 1,941 day columns (wide format)
- `calendar.csv` — 1,969 rows, maps day numbers to dates and events
- `sell_prices.csv` — ~6.8 million rows, weekly price per product per store

In [1]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
from tqdm import tqdm
import os

# Paths
RAW_PATH = Path('../data/raw')
PARQUET_PATH = Path('../data/parquet')
PARQUET_PATH.mkdir(parents=True, exist_ok=True)

# Verify raw files exist
files = ['sales_train_validation.csv', 'calendar.csv', 'sell_prices.csv']
for f in files:
    path = RAW_PATH / f
    size_mb = path.stat().st_size / 1024 / 1024
    print(f"✓ {f:<35} {size_mb:.1f} MB")

✓ sales_train_validation.csv          114.4 MB
✓ calendar.csv                        0.1 MB
✓ sell_prices.csv                     194.0 MB


In [2]:
# Inspect calendar
calendar = pd.read_csv(RAW_PATH / 'calendar.csv')
print("=== CALENDAR ===")
print(f"Shape: {calendar.shape}")
print(f"Columns: {calendar.columns.tolist()}")
print(f"\nSample:")
print(calendar.head(3).to_string())
print(f"\nNull values:")
print(calendar.isnull().sum()[calendar.isnull().sum() > 0])

=== CALENDAR ===
Shape: (1969, 14)
Columns: ['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']

Sample:
         date  wm_yr_wk   weekday  wday  month  year    d event_name_1 event_type_1 event_name_2 event_type_2  snap_CA  snap_TX  snap_WI
0  2011-01-29     11101  Saturday     1      1  2011  d_1          NaN          NaN          NaN          NaN        0        0        0
1  2011-01-30     11101    Sunday     2      1  2011  d_2          NaN          NaN          NaN          NaN        0        0        0
2  2011-01-31     11101    Monday     3      1  2011  d_3          NaN          NaN          NaN          NaN        0        0        0

Null values:
event_name_1    1807
event_type_1    1807
event_name_2    1964
event_type_2    1964
dtype: int64


In [3]:
# Inspect sell prices
print("=== SELL PRICES ===")
# Read just first 5 rows — file is large
prices_sample = pd.read_csv(RAW_PATH / 'sell_prices.csv', nrows=5)
print(f"Columns: {prices_sample.columns.tolist()}")
print(f"\nSample:")
print(prices_sample.to_string())

# Get full shape without loading everything
prices_shape = pd.read_csv(RAW_PATH / 'sell_prices.csv', usecols=['store_id']).shape
print(f"\nFull shape: {prices_shape[0]:,} rows × 4 columns")

# Check unique stores and items
prices_stores = pd.read_csv(RAW_PATH / 'sell_prices.csv', usecols=['store_id'])['store_id'].unique()
print(f"\nStores: {sorted(prices_stores)}")

=== SELL PRICES ===
Columns: ['store_id', 'item_id', 'wm_yr_wk', 'sell_price']

Sample:
  store_id        item_id  wm_yr_wk  sell_price
0     CA_1  HOBBIES_1_001     11325        9.58
1     CA_1  HOBBIES_1_001     11326        9.58
2     CA_1  HOBBIES_1_001     11327        8.26
3     CA_1  HOBBIES_1_001     11328        8.26
4     CA_1  HOBBIES_1_001     11329        8.26

Full shape: 6,841,121 rows × 4 columns

Stores: ['CA_1', 'CA_2', 'CA_3', 'CA_4', 'TX_1', 'TX_2', 'TX_3', 'WI_1', 'WI_2', 'WI_3']


In [4]:
# Inspect sales file — read only first 3 rows to understand structure
print("=== SALES (wide format) ===")
sales_sample = pd.read_csv(RAW_PATH / 'sales_train_validation.csv', nrows=3)
print(f"Columns (first 10): {sales_sample.columns[:10].tolist()}")
print(f"Columns (last 5):   {sales_sample.columns[-5:].tolist()}")
print(f"Total columns:      {len(sales_sample.columns)}")
print(f"\nFirst 3 rows (first 8 columns):")
print(sales_sample.iloc[:, :8].to_string())

# Count total rows without loading
sales_rows = sum(1 for _ in open(RAW_PATH / 'sales_train_validation.csv')) - 1
print(f"\nTotal rows: {sales_rows:,}")

# Unique categories and stores
print(f"\nUnique categories: {sales_sample['cat_id'].unique()}")
print(f"Unique stores: {sales_sample['store_id'].unique()}")

=== SALES (wide format) ===
Columns (first 10): ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1', 'd_2', 'd_3', 'd_4']
Columns (last 5):   ['d_1909', 'd_1910', 'd_1911', 'd_1912', 'd_1913']
Total columns:      1919

First 3 rows (first 8 columns):
                              id        item_id    dept_id   cat_id store_id state_id  d_1  d_2
0  HOBBIES_1_001_CA_1_validation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1       CA    0    0
1  HOBBIES_1_002_CA_1_validation  HOBBIES_1_002  HOBBIES_1  HOBBIES     CA_1       CA    0    0
2  HOBBIES_1_003_CA_1_validation  HOBBIES_1_003  HOBBIES_1  HOBBIES     CA_1       CA    0    0

Total rows: 30,490

Unique categories: ['HOBBIES']
Unique stores: ['CA_1']


## Data Structure Summary

### sales_train_validation.csv
- **Format:** Wide — one row per product-store, one column per day
- **Rows:** 30,490 product-store combinations
- **Columns:** 1,919 (6 metadata + 1,913 day columns d_1 through d_1913)
- **Long format size:** ~58 million rows after melting
- **Join key to calendar:** day column name (d_1, d_2 ... d_1913)

### calendar.csv
- **Rows:** 1,969 days (2011-01-29 through 2016-06-19)
- **Key columns:** `d` (day number), `date`, `wm_yr_wk`, `snap_CA/TX/WI`, `event_name_1/2`
- **Nulls:** event columns mostly null — expected, most days have no events

### sell_prices.csv
- **Rows:** 6,841,121
- **Key columns:** `store_id`, `item_id`, `wm_yr_wk`, `sell_price`
- **Join key to calendar:** `wm_yr_wk`

### Join Strategy
```
sales (long format)
    JOIN calendar ON d = d
    JOIN sell_prices ON store_id + item_id + wm_yr_wk
```

In [5]:
#  Load calendar fully (only 1,969 rows — fits easily in RAM)
calendar = pd.read_csv(RAW_PATH / 'calendar.csv')

# Fill event nulls with 'none' for easier feature engineering later
calendar['event_name_1'] = calendar['event_name_1'].fillna('none')
calendar['event_type_1'] = calendar['event_type_1'].fillna('none')
calendar['event_name_2'] = calendar['event_name_2'].fillna('none')
calendar['event_type_2'] = calendar['event_type_2'].fillna('none')

# Convert date to datetime
calendar['date'] = pd.to_datetime(calendar['date'])

print(f"Calendar loaded: {len(calendar)} rows")
print(f"Date range: {calendar['date'].min()} to {calendar['date'].max()}")
print(f"Columns: {calendar.columns.tolist()}")

Calendar loaded: 1969 rows
Date range: 2011-01-29 00:00:00 to 2016-06-19 00:00:00
Columns: ['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']


In [6]:
# SProcess sell prices in chunks and write to Parquet
PRICES_PARQUET = PARQUET_PATH / 'sell_prices.parquet'

print("Processing sell prices...")
chunks = []
chunk_size = 500_000

for chunk in tqdm(pd.read_csv(RAW_PATH / 'sell_prices.csv', chunksize=chunk_size),
                  total=14, desc="Sell prices chunks"):
    
    # Join to calendar to get date from wm_yr_wk
    chunk = chunk.merge(
        calendar[['wm_yr_wk', 'date', 'month', 'year']].drop_duplicates('wm_yr_wk'),
        on='wm_yr_wk',
        how='left'
    )
    chunks.append(chunk)

# Combine and write to Parquet
prices_df = pd.concat(chunks, ignore_index=True)
prices_df.to_parquet(PRICES_PARQUET, index=False, compression='snappy')

print(f"\nSell prices written to Parquet")
print(f"Shape: {prices_df.shape}")
print(f"Size on disk: {PRICES_PARQUET.stat().st_size / 1024 / 1024:.1f} MB")
del prices_df

Processing sell prices...


Sell prices chunks: 100%|██████████| 14/14 [00:04<00:00,  2.82it/s]



Sell prices written to Parquet
Shape: (6841121, 7)
Size on disk: 4.6 MB


In [8]:
# Process sales data in chunks, melt wide to long, join calendar
SALES_PARQUET = PARQUET_PATH / 'daily_sales.parquet'

# Day columns to melt
day_cols = [f'd_{i}' for i in range(1, 1914)]
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

CHUNK_SIZE = 500
writer = None

print("Processing sales data...")
total_chunks = 30490 // CHUNK_SIZE + 1

for i, chunk in enumerate(tqdm(
    pd.read_csv(RAW_PATH / 'sales_train_validation.csv', chunksize=CHUNK_SIZE),
    total=total_chunks,
    desc="Sales chunks"
)):
    # Melt wide to long
    long = chunk.melt(
        id_vars=id_cols,
        value_vars=day_cols,
        var_name='d',
        value_name='units_sold'
    )

    # Join calendar
    long = long.merge(
        calendar[['d', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year',
                  'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI']],
        on='d',
        how='left'
    )

    # Drop rows with no sales data (products not yet available)
    long = long.dropna(subset=['units_sold'])
    long['units_sold'] = long['units_sold'].astype('int16')

    # Write to Parquet incrementally
    table = pa.Table.from_pandas(long, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(SALES_PARQUET, table.schema, compression='snappy')
    writer.write_table(table)

if writer:
    writer.close()

print(f"\nSales data written to Parquet")
print(f"Size on disk: {SALES_PARQUET.stat().st_size / 1024 / 1024:.1f} MB")

Processing sales data...


Sales chunks:   0%|          | 0/61 [00:00<?, ?it/s]

Sales chunks: 100%|██████████| 61/61 [04:02<00:00,  3.97s/it]


Sales data written to Parquet
Size on disk: 55.7 MB


In [9]:
# Step 4 — Verify Parquet files
print("=== VERIFYING PARQUET FILES ===\n")

# Read sample from sales parquet
sales_sample = pd.read_parquet(SALES_PARQUET)
print(f"Daily sales shape: {sales_sample.shape}")
print(f"Columns: {sales_sample.columns.tolist()}")
print(f"\nSample rows:")
print(sales_sample.head(5).to_string())
print(f"\nDate range: {sales_sample['date'].min()} to {sales_sample['date'].max()}")
print(f"Unique stores: {sorted(sales_sample['store_id'].unique())}")
print(f"Unique categories: {sorted(sales_sample['cat_id'].unique())}")
print(f"Total units sold: {sales_sample['units_sold'].sum():,}")

del sales_sample

=== VERIFYING PARQUET FILES ===

Daily sales shape: (58327370, 19)
Columns: ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'units_sold', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI']

Sample rows:
                              id        item_id    dept_id   cat_id store_id state_id    d  units_sold       date  wm_yr_wk   weekday  wday  month  year event_name_1 event_type_1  snap_CA  snap_TX  snap_WI
0  HOBBIES_1_001_CA_1_validation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1       CA  d_1           0 2011-01-29     11101  Saturday     1      1  2011         none         none        0        0        0
1  HOBBIES_1_002_CA_1_validation  HOBBIES_1_002  HOBBIES_1  HOBBIES     CA_1       CA  d_1           0 2011-01-29     11101  Saturday     1      1  2011         none         none        0        0        0
2  HOBBIES_1_003_CA_1_validation  HOBBIES_1_003  HOBBIES_1  HOBBIES     CA_1       CA  d_

## Ingestion Summary

| File | Raw Size | Parquet Size | Rows |
|---|---|---|---|
| `sales_train_validation.csv` | 114.4 MB | 55.7 MB | 58,327,370 |
| `sell_prices.csv` | 194.0 MB | 4.6 MB | 6,841,121 |
| `calendar.csv` | 0.1 MB | loaded in RAM | 1,969 |

**Total raw:** 308.5 MB → **Total Parquet:** 60.3 MB

### Key decisions
- Sales processed in chunks of 500 rows — never more than ~1M rows in RAM at once
- Wide-to-long transformation done per chunk using `pd.melt`
- Calendar joined during ingestion — date features available in every row
- Sell prices kept separate — joined only when price features are needed
- `units_sold` cast to `int16` — saves ~50% memory vs default int64

### What's on disk

data/parquet/
├── daily_sales.parquet     55.7 MB — 58M rows, sales + calendar features
└── sell_prices.parquet      4.6 MB — 6.8M rows, prices by store/item/week

### Next
`02_demand_forecasting.ipynb` — engineer lag features and train LightGBM
demand forecasting model on FOODS category